# 01 — ODAC23 EDA: minimum H₂O binding energy per MOF

Goal of this notebook: build intuition for the target variable before any modeling.

Questions to answer:

1. What is the distribution of minimum H₂O binding energy across MOFs? Range, mode, tails?
2. How many MOFs are pristine vs defective, and do their distributions differ?
3. How many H₂O configurations were sampled per MOF? Are there MOFs with very few configurations (undersampled, so the 'min' is unreliable)?
4. Are there obvious outliers — values that are either nonphysically extreme or signal a data-cleaning issue?
5. What's our usable sample size after dropping anything suspect?

**Prerequisite**: run `uv run scripts/build_dataset.py` first to produce `data/processed/odac23_min_h2o_per_mof.csv`.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from mofwater.data import ODAC23_PROCESSED_FILENAME, load_odac23, min_h2o_binding_per_mof

# Notebook-wide plotting defaults
plt.rcParams.update({"figure.dpi": 110, "savefig.dpi": 200})

## Load the processed table

If the file doesn't exist yet, run the build script first from a terminal.

In [ ]:
processed_path = Path("data/processed") / ODAC23_PROCESSED_FILENAME

if not processed_path.exists():
    raise FileNotFoundError(
        f"Run `uv run scripts/build_dataset.py` first — expected {processed_path}"
    )

per_mof = pd.read_csv(processed_path)
print(f"Loaded {len(per_mof):,} MOFs")
per_mof.head()

In [ ]:
per_mof.describe()

## Distribution of the target

Sanity expectations for water binding energies:

- A single water molecule in a generic MOF pore should bind in the range of roughly **−0.1 to −0.8 eV** (~−10 to −80 kJ/mol).
- Strongly polar / acidic / open-metal-site MOFs can go more negative, into the −1 to −2 eV range.
- Anything beyond ~−3 eV is suspicious (probably a chemisorption / reaction event, not a normal physisorption configuration).
- Positive values (repulsive) are physically possible but rare for water in microporous solids.

We're looking for: a reasonable shape, no implausible tails, and a sample size large enough to train a model.

In [ ]:
target = per_mof["min_h2o_binding_eV"]

fig, ax = plt.subplots(figsize=(7, 4))
ax.hist(target, bins=80, edgecolor="black", linewidth=0.3)
ax.set_xlabel("Min H$_2$O binding energy per MOF (eV)")
ax.set_ylabel("Count")
ax.set_title(f"ODAC23: min H$_2$O binding energy across {len(per_mof):,} MOFs")
ax.axvline(target.median(), color="red", linestyle="--", label=f"median = {target.median():.2f} eV")
ax.legend()
plt.tight_layout()
plt.show()

print(f"Range: [{target.min():.3f}, {target.max():.3f}] eV")
print(f"Median: {target.median():.3f} eV")
print(f"IQR: [{target.quantile(0.25):.3f}, {target.quantile(0.75):.3f}] eV")

## Pristine vs defective MOFs

ODAC23 includes both pristine MOFs and defective variants (1–16% defect concentration). Defects often create stronger binding sites (under-coordinated metal centers), so we'd expect defective MOFs to skew toward more negative binding energies. Worth confirming.

In [ ]:
if "defective" in per_mof.columns:
    fig, ax = plt.subplots(figsize=(7, 4))
    for label, sub in per_mof.groupby("defective"):
        ax.hist(
            sub["min_h2o_binding_eV"],
            bins=60,
            alpha=0.6,
            label=f"{'defective' if label else 'pristine'} (n={len(sub):,})",
            edgecolor="black",
            linewidth=0.2,
        )
    ax.set_xlabel("Min H$_2$O binding energy per MOF (eV)")
    ax.set_ylabel("Count")
    ax.set_title("Pristine vs defective MOFs")
    ax.legend()
    plt.tight_layout()
    plt.show()

    print(per_mof.groupby("defective")["min_h2o_binding_eV"].describe())
else:
    print("No 'defective' column in the per-MOF table; skipping this section.")

## Configuration coverage per MOF

Each MOF has multiple H₂O configurations sampled. The `min` we computed is the minimum over those configurations — if a MOF has only 1 or 2 configurations sampled, the 'true' min is poorly estimated and we may be misranking it. Look at the distribution to decide on a minimum-coverage cutoff.

In [ ]:
n_cfg = per_mof["n_h2o_configurations"]

fig, ax = plt.subplots(figsize=(7, 4))
ax.hist(n_cfg, bins=range(1, int(n_cfg.max()) + 2), edgecolor="black", linewidth=0.3)
ax.set_xlabel("Number of H$_2$O configurations sampled")
ax.set_ylabel("Count of MOFs")
ax.set_title("Per-MOF H$_2$O configuration coverage")
plt.tight_layout()
plt.show()

print(n_cfg.describe())
print()
print(f"MOFs with <= 2 configurations: {(n_cfg <= 2).sum():,}")
print(f"MOFs with <= 5 configurations: {(n_cfg <= 5).sum():,}")

## Outlier check

Are there values outside the physically reasonable range (e.g. more negative than −3 eV, or positive)?

In [ ]:
very_negative = per_mof[per_mof["min_h2o_binding_eV"] < -3.0]
positive = per_mof[per_mof["min_h2o_binding_eV"] > 0.0]

print(f"MOFs with min binding < -3 eV (suspicious): {len(very_negative)}")
print(f"MOFs with positive min binding (repulsive): {len(positive)}")

if len(very_negative):
    print("\nTop 10 most-negative:")
    print(very_negative.nsmallest(10, "min_h2o_binding_eV").to_string(index=False))

## Observations and next steps

Fill in as you actually run this notebook:

1. **Distribution shape**: ...
2. **Pristine vs defective**: ...
3. **Configuration coverage**: ...
4. **Outliers**: ...
5. **Sample size after sensible filtering**: ...

**Decisions for the modeling phase**:

- Minimum-configurations cutoff: ?
- Outlier handling (drop / clip / keep): ?
- Include defective MOFs in train? In test? Or treat them as a separate OOD set?
- Split strategy: hold out by MOF ID (always), and by ... (composition? topology? cluster in feature space)?

Once these are decided, write them up as a new entry in `notes/logbook.md` and move on to feature computation.